# AGender - Step 1: cleaning rules and fixed splits (CPU)

- **LAT:** applies the agreed cleaning rules (`agender/lat.py`): the six per-app session counts become one
  (`SESSIONS`, the largest of the six, because they copy each other), `NEED_CUSTOMER_SERVICE` is dropped,
  device age becomes years, missing device fields become `unknown`, the 913 subscribers without data use are kept.
- **Copenhagen:** one row per student with known gender, from the call and SMS logs (`agender/copenhagen.py`).
- **Splits:** train 60 % / validation 10 % / calibration 10 % / test 20 %, stratified (LAT: gender x six age
  groups; Copenhagen: gender), five repeats with the seeds 42, 7, 123, 2024, 99.
- **Check:** the splits made here must be identical to the ones in the repository (`splits/`).

**Kaggle settings:** Accelerator = **None** (CPU). Internet = Off is fine.
**Inputs to add:** `agender-code` and `agender-step0` (the saved output of Step 0).
**Expected run time:** under a minute. The output folder can be saved as the dataset `agender-step1`.

In [ ]:
import os
import sys
from pathlib import Path

# The AGender code is attached as a Kaggle dataset. Find the `agender` package
# wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('audit.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The agender-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> agender-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if package.parent.name == 'agender' and (package.parent / '__init__.py').exists():
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder; link the files under the name `agender`.
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'agender').exists():
        os.symlink(package.parent, code_root / 'agender')

sys.path.insert(0, str(code_root))
import agender
print('AGender code loaded from', Path(agender.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/agender_outputs/step1')
TIME_BUDGET_MINUTES = 30

In [ ]:
import json

from agender import environment, step1
from agender.runner import StepRunner

runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {}

# Both files come from the saved output of Step 0 (agender-step0). The raw copenhagen-networks-study
# dataset may also be attached; we skip it and use the copy Step 0 checked, in data/copenhagen/.
# Kaggle unpacks zip files when an output is saved as a dataset, so look for the zip or the csv inside it.
lat_zip = find_file('lat_s1.zip', INPUT) or find_file('age-gender-pred-dataset.csv', INPUT)
step0_copies = sorted(p for p in INPUT.rglob('genders.csv') if p.parent.name == 'copenhagen')
genders = step0_copies[0] if step0_copies else None
print('LAT file:', lat_zip)
print('Copenhagen folder:', genders.parent if genders else None)
if lat_zip is None or genders is None:
    attached = sorted(str(p) for p in INPUT.glob('*/*/*'))[:20]
    raise RuntimeError('Attach the saved output of Step 0 (Add Input -> Your Work -> the Step 0 notebook, or the '
                       'dataset agender-step0): it holds data/lat_s1.zip (or its unpacked csv) and data/copenhagen/. '
                       f'Attached now: {attached}')

summary['environment'] = runner.run('measure the machine', environment.collect)
# read_copenhagen expects <folder>/copenhagen/*.csv
summary['tables'] = runner.run('clean data and make splits', step1.write_all, lat_zip, genders.parent.parent, OUTPUT)
summary['same_as_repository'] = runner.run('compare with splits/ in the code', step1.compare_with_shipped,
                                           OUTPUT, code_root / 'splits')
summary['steps'] = runner.steps
with open(OUTPUT / 'step1_summary.json', 'w') as f:
    json.dump(summary, f, indent=1, default=str)
print(json.dumps({k: summary[k] for k in ('same_as_repository', 'steps')}, indent=1, default=str))
print('Failed or skipped steps:', runner.failed() or 'none')